In [ ]:
using Pkg
Pkg.build("GeneralizedPerturbedEquilibrium")

In [ ]:
using Pkg
Pkg.activate("/Users/bursche/Documents/GitHub/JPEC_BCRIT")
Base.active_project()

using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: Analysis
using Printf

using Plots
default(
    fontfamily="Georgia",
    margin=12Plots.mm,
    size=(800, 500),
    dpi=150
)

In [ ]:
h5path = "/Users/bursche/Documents/GitHub/JPEC_BCRIT/examples/DIIID-like_SLAYER_and_bcrit_example/gpec.h5"

In [ ]:
using HDF5

h5open(h5path, "r") do file
    println("Top-level keys:")
    println(collect(keys(file)))

    if haskey(file, "Tearing")
        println("\nKeys in Tearing:")
        println(collect(keys(file["Tearing"])))

        if haskey(file["Tearing"], "CriticalResonantField")
            println("\nKeys in Tearing/CriticalResonantField:")
            println(collect(keys(file["Tearing"]["CriticalResonantField"])))
        end
    else
        println("No Tearing group found.")
    end
end

In [ ]:
# Print the contents of the Tearing/CriticalResonantField group
h5open(h5path, "r") do file
    if haskey(file, "Tearing") && haskey(file["Tearing"], "CriticalResonantField")
        println("\nContents of Tearing/CriticalResonantField:")
        # Iterate over the keys in the Tearing/CriticalResonantField group and print their names and types
        for key in keys(file["Tearing"]["CriticalResonantField"])
            dataset = file["Tearing"]["CriticalResonantField"][key]
            println("Key: $key, Type: $(typeof(dataset))")
            # print data 
            if isa(dataset, HDF5.Dataset)
                data = read(dataset)
                println("Data: $data")
            end
        end
    else
        println("No Tearing/CriticalResonantField group found.")
    end
    # Print bcrit data as .2e (in one line)
    println("\nBcrit data: $(join([@sprintf("%.2e", x) for x in read(file["Tearing"]["CriticalResonantField"]["br_crit"])], " "))")
end



In [ ]:
println(keys(h5open(h5path, "r")["Tearing"]["CriticalResonantField"]["Scan"]["surface_1"]))

In [ ]:
function bcrit_diag_plots(Δs, Qs, bal, name)

    p1 = plot(Qs, imag.(Δs), label="Im(Δ)", lw=2)
    #plot!(p1, Qs, real.(Δs), label="Re(Δ)", lw=2)
    xlabel!(p1, "Q")
    ylabel!(p1, "Δ")
    title!(p1, "Inner-layer Δ(Q) - $name")

    p2 = plot(Qs, real.(bal), label="Re(balance)", lw=2)
    plot!(p2, Qs, imag.(bal), label="Im(balance)", lw=2)
    xlabel!(p2, "Q")
    ylabel!(p2, "balance")
    title!(p2, "2P(Q0-Q)/jxb - $name")

    plot(p1, p2, layout=(2,1), size=(500, 700))


end

function plot_all_vs_rational_q(h5path)
    h5open(h5path, "r") do file
        tearing = file["Tearing"]
        crf = tearing["CriticalResonantField"]
        scan = crf["Scan"]

        rational_q = read(tearing["PerSurface"]["rational_q"])
        bcrit = read(crf["br_crit"])
        bal = [read(scan["surface_$(i)"]["balance"]) for i in eachindex(rational_q)]
        P = [read(scan["surface_$(i)"]["P"]) for i in eachindex(rational_q)]
        lu = [read(scan["surface_$(i)"]["lu"]) for i in eachindex(rational_q)]
        sval = [read(scan["surface_$(i)"]["sval"]) for i in eachindex(rational_q)]
        Q0 = [read(scan["surface_$(i)"]["Q0"]) for i in eachindex(rational_q)]
        max_balance = maximum.(bal)

        fig = plot(layout=(3, 2), size=(1000, 1200))
        plot!(fig[1], rational_q, bcrit; marker=:circle, lw=2, xlabel="Rational Surface (q)", ylabel="Critical Resonant Field (T)", legend=false)
        plot!(fig[2], rational_q, max_balance; marker=:circle, lw=2, xlabel="Rational Surface (q)", ylabel="Maximum (Torque Balance)", legend=false)
        plot!(fig[3], rational_q, P; marker=:circle,  lw=2, xlabel="Rational Surface (q)", ylabel="P", legend=false)
        plot!(fig[4], rational_q, lu; marker=:circle, lw=2, xlabel="Rational Surface (q)", ylabel="Lundquist Number", legend=false)
        plot!(fig[5], rational_q, sval; marker=:circle, lw=2, xlabel="Rational Surface (q)", ylabel="Magnetic Shear", legend=false)
        plot!(fig[6], rational_q, Q0; marker=:circle, lw=2, xlabel="Rational Surface (q)", ylabel="Q0", legend=false)

        display(fig)
        #return fig
    end
end

plot_all_vs_rational_q(h5path)

In [ ]:
using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: Analysis
using GeneralizedPerturbedEquilibrium.InnerLayer: InnerLayerModel, solve_inner, GGJModel, GGJParameters,
    SLAYERModel, SLAYERParameters, slayer_parameters
using GeneralizedPerturbedEquilibrium.Tearing.CriticalResonantField: TorqueBalance, torque_balance_value, torque_balance_scan

surface_num = 5
h5open(h5path, "r") do file
    sval_r = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["sval"])
    Q0 = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["Q0"])
    m = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["m"])
    n = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["n"])
    qval = m/n
    Q_e = read(file["Tearing"]["PerSurface"]["Q_e"])[surface_num]
    Q_i = read(file["Tearing"]["PerSurface"]["Q_i"])[surface_num]
    lu = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["lu"])
    rs = read(file["Tearing"]["PerSurface"]["rs"])[surface_num]
    P = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["P"])

    p = slayer_parameters( # Call slayer_parameters function to instantiate slab parameters
        n_e=1e19, t_e=1e3, t_i=1e3,
        omega=0.0, omega_e=40, omega_i=-20,
        qval=qval, sval_r=sval_r, bt=2.0, rs=rs, R0=3.0, mu_i=2.0, zeff=1.0,
        chi_perp=1.0, chi_tor=1.0, m=2, n=1
    )
    p = SLAYERParameters( # Modify the created SLAYERParameters struct to modify electron and ion diamagnetic resonance frequencies
        ising=p.ising,
        m=p.m, n=p.n,
        tau=p.tau, lu=p.lu, c_beta=p.c_beta, D_norm=p.D_norm,
        P_perp=p.P_perp, P_tor=p.P_tor,
        Q_e=Q_e, Q_i=Q_i, iota_e=p.iota_e,
        tauk=p.tauk, tau_r=p.tau_r, delta_n=p.delta_n,
        rs=p.rs, R0=p.R0, bt=p.bt, sval_r=p.sval_r,
        dr_val=p.dr_val, dgeo_val=p.dgeo_val,
        eta=p.eta, d_beta=p.d_beta,
        dc_tmp=p.dc_tmp, dc_type=p.dc_type
    )

    tb = TorqueBalance( # Generate a TorqueBalance struct using the default (Fitzpatrick) SLAYERModel and the modified SLAYERParameters
        SLAYERModel(;),
        p, # Slayer Parameters struct
        Q0, # Normalized rotation Q
        P, # Magnetic Prandtl number
        p.lu, # Lundquist Number
        p.sval_r # r-based magnetic shear
    )
    Qmin = -.2
    Qmax = .2
    Qs, bal, Qpeak, brcrit, Qpeak_ind, Δs = torque_balance_scan(tb,Qmin=Qmin, Qmax=Qmax, n=1000) # Run actual torque balance scan, finds inner-layer delta across Q vals


    jxbs = [-imag(1.0 / (d + 1e-2)) for d in Δs]
    T_VISC = [2.0 * tb.P * (tb.Q0 - q) for q in Qs] #(q, jxb) in zip(Qs, jxbs)]
    T_EM = -imag(1.0 ./ (Δs .+ 1e-2)) *brcrit^2*tb.lu/(tb.sval^2/2)
    i = Qpeak_ind
    println(T_VISC[i]/(tb.lu * tb.sval^2/2 * (brcrit)^2 * jxbs[i]))
    println(T_VISC[i] / (tb.lu * tb.sval^2/2 * (brcrit * p.bt)^2 * jxbs[i]))
    println(T_VISC[i] / ( (brcrit / p.bt)^2 * jxbs[i]))

    #T_EM ∝ S ξ̂ (br/Bφ)² Im[-Δ̂(Q)⁻¹]
    #T_visc ∝ 2 P (Q0 − Q)


    xmi = Qmin
    xma = Qmax

    """
    p1 = plot(Qs, imag.(Δs), label="Im(Δ)", lw=2, xlim=(xmi, xma))
    #plot!(p1, Qs, real.(Δs), label="Re(Δ)", lw=2)
    xlabel!(p1, "Q")
    ylabel!(p1, "Δ")
    #title!(p1, "Inner-layer Δ(Q)")
    """

    p1 = plot( Qs, T_VISC, label="T_V", lw=2, xlim=(xmi, xma))#,ylim=(-1000,1000))
    plot!(p1, Qs, T_EM, label="T_EM", lw=2)
    vline!([Qpeak], label="peak Q", linestyle=:dash)
    xlabel!(p1, "Q")
    ylabel!(p1, "Torque")

    #p2 = plot(Qs, jxbs, label="jxb", lw=2)
    p2 = plot( Qs, T_VISC, label="T_V", lw=2, xlim=(xmi, xma))#,ylim=(0,500))
    plot!(p2, Qs, T_EM, label="T_EM", lw=2)
    vline!([Qpeak], label="peak Q", linestyle=:dash)
    xlabel!(p2, "Q")
    ylabel!(p2, "Torque")

    p3 = plot(Qs, real.(bal), label="Re(balance)", lw=2, xlim=(xmi, xma),ylim=(-1e-2,1e-2))
    plot!(p3, Qs, imag.(bal), label="Im(balance)", lw=2)
    ylabel!(p3, "Torque Balance")
    xlabel!(p3, "Q")
    vline!([Qpeak], label="Max Balance", linestyle=:dash)

    # find index, q val and bal val of the q closest to Q_e
    println("Q_e: ", -tb.params.Q_e)
    idx_closest_Q_e = argmin(abs.(Qs .+ tb.params.Q_e))
    q_closest_Q_e = Qs[idx_closest_Q_e]
    bal_closest_Q_e = bal[idx_closest_Q_e]
    println("Closest Q to Q_e: ", q_closest_Q_e, " with balance value: ", bal_closest_Q_e, " at index: ", idx_closest_Q_e)
    println("B crit: ", brcrit)
    vline!([-tb.params.Q_e], label="Q_e", linestyle=:dashdot, color=:orange)
    vline!([q_closest_Q_e], label="Closest Q to Q_e", linestyle=:dashdot, color=:purple)

    #plot(p1, p2, p3, layout=(3,1), size=(800, 1000))
    plot(p3, layout=(1,1))#, size=(800, 1000))

end

In [ ]:
# plot jxb vs Qs for surface 1
# jxbs = [-imag(1.0 / (d + 1e-2)) for d in Δs]
surface_num = 5
h5open(h5path, "r") do file
    Δs = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["delta"])
    Qs = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["Q"])
    bal = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["balance"])
    jxbs = [-imag(1.0 / (d + 1e-2)) for d in Δs]
    aplot = plot(Qs, jxbs, label="jxb", lw=2)
    xlabel!("Q")
    ylabel!("jxb")
    title!("jxb vs Q for surface $(surface_num)")
    println("max of balance: ", maximum(bal))
    println("max of jxb: ", maximum(jxbs))
    println("bcrit: ", read(file["Tearing"]["CriticalResonantField"]["br_crit"]))
    println("Qpeak: ", read(file["Tearing"]["CriticalResonantField"]["Qpeak"]))
    println("Q_e:   ", read(file["Tearing"]["PerSurface"]["Q_e"]))
    # plot balance vs q with bcrit as vertical line
    p2 = plot(Qs, real.(bal), label="Re(balance)", lw=2,xlim=(-.2,.2),ylim=(-1e-2,1e-2))
    plot!(p2, Qs, imag.(bal), label="Im(balance)", lw=2)
    xlabel!(p2, "Q")
    ylabel!(p2, "balance")
    title!(p2, "2P(Q0-Q)/jxb - surface $(surface_num)")
    #vline!(p2, [read(file["Tearing"]["CriticalResonantField"]["br_crit"])], label="bcrit", lw=2, lc=:red)
    # add Q_e as vertical line
    vline!(p2, [read(file["Tearing"]["PerSurface"]["Q_e"])[surface_num]], label="Q_e", lw=2, lc=:green)
    vline!(p2, [read(file["Tearing"]["PerSurface"]["Q_i"])[surface_num]], label="Q_i", lw=2, lc=:blue)
    vline!(p2, [read(file["Tearing"]["CriticalResonantField"]["Qpeak"])[surface_num]], label="Qpeak", lw=2, lc=:orange)
    #display(aplot)
    display(p2)

end

"""
Local maxima indices: [104, 103] with values: [80.15320925995013, 79.66428209429431] and corresponding Qs: [0.35175879396984927, 0.25125628140703515] closest Q to Q_e: -0.35175879396984927 closest Q to Q_i: 0.35175879396984927
Warning: The first local maximum may correspond to an electron or ion diamagnetic resonance. Selecting the second local maximum instead.
Selected local maximum index: 103 with value: 79.66428209429431 and corresponding Q: 0.25125628140703515
"""

In [ ]:
# Plot bcrit vs q surface

# print all contents of bcrit /scan /surface 1/
h5open(h5path, "r") do file
    bcrit = file["Tearing"]["CriticalResonantField"]

    for i in 1:6
        ss = "surface_$i"

        delta  = read(bcrit["Scan"][ss]["delta"])
        Q      = read(bcrit["Scan"][ss]["Q"])
        balance = read(bcrit["Scan"][ss]["balance"])

        display(bcrit_diag_plots(delta, Q, balance, "surface_$i"))
    end
end

In [ ]:
p_eq = Analysis.Equilibrium.plot_equilibrium_summary(h5path)
p_ffs = Analysis.ForceFreeStates.plot_ffs_summary(h5path)
#p_pe = Analysis.PerturbedEquilibrium.plot_perturbed_equilibrium_summary(h5path)

display(p_eq)
display(p_ffs)
#display(p_pe)